# 🚀 QTrader Analyst Platform – Colab Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/qtrader/blob/main/notebooks/Colab_Quickstart.ipynb)

**Three roles, one notebook**: configure below and all cells adapt automatically.

| Role | You are... | Notebooks opened |
|---|---|---|
| `analyst` | Quant Analyst | EDA, Backtest, Risk reports |
| `researcher` | Quant Researcher | Feature Lab, Regime Lab, ML Experiment |
| `trader` | Quant Trader | Live Monitor, Execution Audit |

---

## ⚙️ Configuration – Edit this cell

In [1]:
# ──────────────────────────────────────────
# 📌 REQUIRED: Set your Git repository URL
# ──────────────────────────────────────────
REPO_URL = "https://github.com/namhoangdev31/qtrader.git"   # e.g. "https://github.com/yourname/qtrader.git"
BRANCH   = "main"

# ──────────────────────────────────────────
# 👤 ROLE: analyst | researcher | trader
# ──────────────────────────────────────────
ROLE = "trader"

# ──────────────────────────────────────────
# 💾 Google Drive (optional)
# ──────────────────────────────────────────
USE_GDRIVE   = False   # Mount Google Drive for persistent datalake / reports
GDRIVE_PATH  = "/content/drive/MyDrive/qtrader"  # Path inside your Drive

## 📦 Setup – Clone Repo & Install

In [2]:
import os, subprocess, sys

# --- Google Drive ---
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_PATH, exist_ok=True)
    print(f"✅ Google Drive mounted → {GDRIVE_PATH}")

# --- Clone or update repo ---
REPO_DIR = "/content/qtrader_repo"
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    result = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    print("✅ Repository cloned.")
else:
    print("Pulling latest changes...")
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)

os.chdir(REPO_DIR)

# --- Install dependencies ---
print("Installing dependencies...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.[analyst]', '-q'])
print("✅ QTrader installed.")

Cloning repository...


RuntimeError: git clone failed:
fatal: could not create leading directories of '/content/qtrader_repo': Read-only file system


## 🎯 Role-Based Session Initialisation

In [ ]:
from qtrader.output.analyst import AnalystSession, RoleContext

session = AnalystSession(role=ROLE)
session.info()

# If using Google Drive as datalake, configure the path
if USE_GDRIVE:
    import os
    os.environ['QTRADER_DATALAKE_PATH'] = GDRIVE_PATH
    print(f"📂 Datalake → {GDRIVE_PATH}")

## 🔄 Quickstart Workflow

Run the cells below for a quick end-to-end demo. Then open the role-specific notebooks in `notebooks/<role>/` for deep-dive analysis.

In [ ]:
import polars as pl

symbol = "BTC-USD"
timeframe = "1d"

# Load data (datalake → synthetic fallback)
try:
    df = session.load_ohlcv(symbol, timeframe)
    print(f"Loaded {len(df)} rows from datalake")
except Exception:
    df = session.sample_ohlcv(symbol="BTC", days=365)
    print(f"Using synthetic data: {len(df)} rows")

# Compute returns & rolling features
df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 21])
df = df.drop_nulls()
df.head(5)

In [ ]:
# Simple momentum signal
df = df.with_columns(
    pl.when(pl.col('close') > pl.col('sma_21')).then(1).otherwise(-1).alias('signal')
)

bt = session.run_vector_backtest(df, signal_col='signal')
metrics = session.compute_extended_metrics(bt['equity_curve'])

import json
print(json.dumps({k: round(float(v), 4) if isinstance(v, float) else v for k, v in metrics.items()}, indent=2))

In [ ]:
import matplotlib.pyplot as plt

equity = bt['equity_curve'].to_numpy()

fig, ax = plt.subplots(figsize=(12, 4), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
for sp in ax.spines.values(): sp.set_color('#334155')
ax.tick_params(colors='#94a3b8')
ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)
ax.plot(equity, color='#34d399', linewidth=1.5)
ax.set_title(f'QTrader Equity Curve – {symbol} Momentum', color='#e2e8f0')
ax.set_xlabel('Period', color='#94a3b8')
ax.set_ylabel('Equity (normalised)', color='#94a3b8')
plt.tight_layout()
plt.show()

In [ ]:
# Export HTML report to local Colab runtime (or Google Drive if mounted)
report_path = (f"{GDRIVE_PATH}/reports/colab_{symbol.replace('-','_')}_quickstart.html"
               if USE_GDRIVE else f"/content/qtrader_quickstart.html")

out = session.export_report(
    title=f'QTrader Quickstart – {ROLE.title()} | {symbol}',
    sections={
        'Role': f'Running as: **{ROLE}**',
        'Performance Metrics': metrics,
        'Equity Curve': bt['equity_curve'],
    },
    path=report_path,
)
print(f'✅ Report saved: {out}')

# Download from Colab
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass  # Not running in Colab – report saved locally

---
## 📓 Next Steps – Role-Specific Notebooks

After this quickstart, open the deep-dive notebooks for your role:

**Analyst**: `notebooks/analyst/01_EDA_Report.ipynb`, `02_Backtest_Report.ipynb`, `03_Risk_Report.ipynb`

**Researcher**: `notebooks/researcher/01_Feature_Lab.ipynb`, `02_Regime_Lab.ipynb`, `03_ML_Experiment.ipynb`

**Trader**: `notebooks/trader/01_Live_Monitor.ipynb`, `02_Execution_Audit.ipynb`